# Tugas 5 — Feature Selection dan Hyperparameter Tuning
## Dataset UNSW-NB15 | Bengkel Koding — Google Colab

**Kelanjutan Tugas 4: Validation dan Sampling.** Target tetap `label`: **0 = Normal**, **1 = Attack**. Model utama tetap **Logistic Regression**, dengan preprocessing dan opsi sampling seperti minggu lalu.

**Tujuan pembelajaran:**
1. Mengetahui perbedaan **Feature Selection** (memilih fitur input yang relevan) dan **Hyperparameter Tuning** (mencari pengaturan model).
2. Membandingkan model **tanpa Feature Selection** dan **dengan Feature Selection**.
3. Melakukan `GridSearchCV` menggunakan `StratifiedKFold`, lalu mengukur hasil terbaik pada data validasi dan test.
4. Menyimpan tabel hasil, fitur terpilih, confusion matrix, dan penjelasan.

**Prinsip penting:** fitur dipilih, imputasi dilakukan, scaler di-*fit*, dan sampling diterapkan **hanya pada bagian training di setiap fold**. Data validasi/test tidak dipakai untuk menentukan fitur ataupun hyperparameter.

> **Cara pakai:** Unggah `.ipynb` ini ke Google Colab → pilih **Runtime → Run all**. Dataset diambil dari Kaggle `darkn3t/unsw-nb15-training-set-csv`, sebagaimana tugas sebelumnya. Semua kode mempunyai penjelasan pada sel Markdown tepat sebelumnya. Nilai hasil bukan angka asumsi; baru muncul setelah notebook dijalankan.

## A. Persiapan
### Penjelasan Sel 1 — Menyiapkan library
`scikit-learn` digunakan untuk model dan validasi, `imbalanced-learn` untuk sampling (jika dipakai), `pandas` untuk tabel, `matplotlib` untuk grafik. Instalasi hanya dilakukan jika library belum tersedia agar Colab tidak mengubah versi yang sudah terpasang tanpa perlu.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('imblearn') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'imbalanced-learn'])

print('Library pendukung siap.')

### Penjelasan Sel 2 — Import dan konfigurasi
`RANDOM_STATE=42` supaya pembagian data dapat diulang dengan hasil sama. `DATA_SOURCE='auto'` mencari data di Kaggle/Colab atau mengunduhnya dengan `kagglehub`. Ubah ke `'manual'` hanya jika ingin mengunggah CSV sendiri. Semua hasil disimpan di folder kerja `Hasil_Tugas_5_UNSW_NB15`.

In [ ]:
import json, sys, warnings, shutil
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import sklearn

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, balanced_accuracy_score,
    precision_score, recall_score, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report)
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib

RANDOM_STATE = 42
DATA_SOURCE = 'auto'  # 'auto' atau 'manual'
KAGGLE_HANDLE = 'darkn3t/unsw-nb15-training-set-csv'
KAGGLE_INPUT_PATH = Path('/kaggle/input/datasets/darkn3t/unsw-nb15-training-set-csv')
MANUAL_DATA_DIR = Path('/content') if Path('/content').is_dir() else Path.cwd()
FOLDS = 3

WORK_DIR = Path('/content') if Path('/content').is_dir() else (Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd())
OUTPUT_DIR = WORK_DIR / 'Hasil_Tugas_5_UNSW_NB15'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Python:', sys.version.split()[0])
print('scikit-learn:', sklearn.__version__)
print('Folder hasil:', OUTPUT_DIR)

## B. Membaca dataset yang sama
### Penjelasan Sel 3 — Menemukan file CSV
Di Kaggle, lokasi folder dapat berupa `/kaggle/input/...`. Di **Google Colab**, path tersebut biasanya tidak ada. Karena itu kode akan mengunduh dataset publik melalui `kagglehub`. Jika unduhan gagal, gunakan `DATA_SOURCE='manual'` dan unggah satu CSV atau dua CSV training/testing.

Nama file **tidak otomatis dipercaya** sebagai petunjuk isi; ukuran dan distribusi label diperiksa di sel berikutnya.

In [ ]:
def cari_csv(folder):
    return sorted(p for p in folder.rglob('*.csv')
                  if p.is_file() and OUTPUT_DIR not in p.parents)

if DATA_SOURCE == 'manual':
    dataset_root = MANUAL_DATA_DIR
    if not cari_csv(dataset_root):
        try:
            from google.colab import files
            print('Unggah satu CSV UNSW-NB15 atau pasangan CSV training/testing:')
            files.upload()
        except ImportError:
            raise FileNotFoundError(f'Taruh CSV pada folder {MANUAL_DATA_DIR}')
else:
    candidates = [KAGGLE_INPUT_PATH, Path('/kaggle/input/unsw-nb15-training-set-csv')]
    found = next((p for p in candidates if p.is_dir() and cari_csv(p)), None)
    if found is not None:
        dataset_root = found
    else:
        if importlib.util.find_spec('kagglehub') is None:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
        import kagglehub
        try:
            dataset_root = Path(kagglehub.dataset_download(KAGGLE_HANDLE))
        except Exception as exc:
            raise RuntimeError('Kaggle gagal diunduh. Gunakan DATA_SOURCE="manual", lalu unggah CSV.') from exc

csv_files = cari_csv(dataset_root)
if not csv_files:
    raise FileNotFoundError('Tidak menemukan file CSV UNSW-NB15.')

def namafile(p):
    return p.stem.lower().replace('_', '').replace('-', '').replace(' ', '')

if len(csv_files) == 1:
    TRAIN_FILE, TEST_FILE = csv_files[0], None
else:
    training = [p for p in csv_files if 'train' in namafile(p)]
    testing = [p for p in csv_files if 'test' in namafile(p)]
    if len(csv_files) == 2 and len(training) == 1 and len(testing) == 1:
        TRAIN_FILE, TEST_FILE = training[0], testing[0]
    else:
        raise ValueError('Nama CSV ambigu; gunakan folder khusus satu CSV atau sepasang file training/testing.')

print('Sumber:', dataset_root)
print('CSV utama:', TRAIN_FILE.name)
print('CSV testing:', TEST_FILE.name if TEST_FILE else 'Tidak ada; akan dibuat holdout')
for p in csv_files:
    print(' -', p.name, 'ukuran', round(p.stat().st_size / 1e6, 2), 'MB')

### Penjelasan Sel 4 — Validasi label dan struktur data
Baca data, pastikan kolom `label` berisi hanya **0 atau 1**, lalu tampilkan jumlah baris, missing value, duplikat, dan distribusi Normal/Attack. Baris dengan label tidak valid dikeluarkan karena tidak mempunyai jawaban benar untuk *supervised learning*. Duplikat trafik **tidak otomatis dihapus** karena kejadian serupa bisa benar-benar terjadi.

In [ ]:
def baca_unsw(path):
    frame = pd.read_csv(path, encoding='utf-8-sig', low_memory=False)
    frame.columns = frame.columns.astype(str).str.replace('\ufeff', '', regex=False).str.strip()
    if frame.columns.duplicated().any():
        raise ValueError(f'Terdapat nama kolom duplikat pada {path.name}')
    if 'label' not in frame.columns:
        raise ValueError(f'Kolom label tidak ada pada {path.name}')
    target = pd.to_numeric(frame['label'], errors='coerce')
    valid = target.isin([0, 1])
    n_reject = int((~valid).sum())
    frame = frame.loc[valid].copy().reset_index(drop=True)
    frame['label'] = target.loc[valid].to_numpy().astype('int8')
    if set(frame['label'].unique()) != {0, 1}:
        raise ValueError(f'{path.name} harus memuat kedua kelas 0 dan 1')
    return frame, n_reject

source_train, rejected_train = baca_unsw(TRAIN_FILE)
source_test, rejected_test = baca_unsw(TEST_FILE) if TEST_FILE else (None, 0)

rows = []
for name, data, rejected in [('CSV utama', source_train, rejected_train),
                             ('CSV test', source_test, rejected_test)]:
    if data is not None:
        rows.append({'Sumber':name,'Baris':len(data),'Kolom':data.shape[1],
                     'Normal (0)':int((data.label==0).sum()),
                     'Attack (1)':int((data.label==1).sum()),
                     'Missing cells':int(data.isna().sum().sum()),
                     'Duplikat persis':int(data.duplicated().sum()),
                     'Label invalid dibuang':rejected})
quality_report = pd.DataFrame(rows)
display(quality_report)
display(source_train.head(3))

## C. Melanjutkan Validation minggu keempat
### Penjelasan Sel 5 — Split yang konsisten
Seperti Tugas 4, gunakan `stratify=label` agar persentase Normal/Attack tetap mirip. Jika **1 CSV**, pembagiannya menjadi sekitar **64% train, 16% validation, 20% test**. Jika **2 CSV**, test memakai CSV khusus dan CSV training dibagi 80% train internal/20% validation.

**Validation** membantu melihat hasil pengembangan, sedangkan **test** disimpan untuk pemeriksaan akhir. Data test tidak digunakan pada proses `GridSearchCV`.

In [ ]:
if source_test is None:
    development_df, test_df = train_test_split(
        source_train, test_size=0.20, stratify=source_train['label'],
        random_state=RANDOM_STATE)
    split_method = 'single_csv_64_16_20_stratified'
else:
    development_df, test_df = source_train, source_test
    split_method = 'two_csv_train_val_and_external_test'

train_df, val_df = train_test_split(
    development_df, test_size=0.20, stratify=development_df['label'],
    random_state=RANDOM_STATE)
train_df, val_df, test_df = [x.reset_index(drop=True) for x in (train_df,val_df,test_df)]

split_report = pd.DataFrame([{
    'Bagian':name,'Jumlah':len(data),
    'Normal (0)':int((data.label==0).sum()),
    'Attack (1)':int((data.label==1).sum()),
    '% Attack':round(100*data.label.mean(),2)
} for name,data in [('Train',train_df),('Validation',val_df),('Test',test_df)]])
display(split_report)
assert all(set(d.label.unique())=={0,1} for d in (train_df,val_df,test_df))

### Penjelasan Sel 6 — Menyiapkan X dan y
`X` adalah fitur trafik jaringan, `y` adalah target `label`. Kolom `attack_cat` dikeluarkan untuk menghindari **target leakage** (informasi kategori serangan membocorkan jawaban), sedangkan `id` adalah pengenal baris, bukan sinyal trafik yang ingin dipelajari.

Fitur numerik diubah ke angka; nilai tidak terbaca menjadi `NaN`. Fitur kategori (`proto`, `service`, `state`) dibersihkan sebagai teks. Penanganan missing dilakukan **di dalam pipeline**, bukan langsung pada seluruh data.

In [ ]:
TARGET = 'label'
EXCLUDE = {'label', 'id', 'attack_cat'}
feature_cols = [c for c in train_df.columns if c not in EXCLUDE]
for label_split, data in [('Validation',val_df),('Test',test_df)]:
    missing_cols = sorted(set(feature_cols) - set(data.columns))
    if missing_cols:
        raise ValueError(f'Fitur tidak ada pada {label_split}: {missing_cols}')

cat_cols = [c for c in feature_cols
            if c in {'proto','service','state'} or not pd.api.types.is_numeric_dtype(train_df[c])]
num_cols = [c for c in feature_cols if c not in cat_cols]

def siapkan_X(frame):
    X = frame[feature_cols].copy()
    for col in num_cols:
        X[col] = pd.to_numeric(X[col], errors='coerce').replace([np.inf,-np.inf], np.nan)
    for col in cat_cols:
        s = X[col].astype('string').str.strip()
        s = s.mask(s.isin(['', '?', '??', 'N/A', 'NA', 'NULL', 'null']))
        X[col] = s.astype(object).where(s.notna(), np.nan)
    return X

X_train, X_val, X_test = [siapkan_X(d) for d in (train_df,val_df,test_df)]
y_train, y_val, y_test = [d[TARGET].astype(int).reset_index(drop=True)
                          for d in (train_df,val_df,test_df)]

empty_cols = X_train.columns[X_train.isna().all()].tolist()
if empty_cols:
    warnings.warn(f'Kolom kosong total di training dikeluarkan: {empty_cols}')
    X_train = X_train.drop(columns=empty_cols)
    X_val = X_val.drop(columns=empty_cols)
    X_test = X_test.drop(columns=empty_cols)
    num_cols = [c for c in num_cols if c not in empty_cols]
    cat_cols = [c for c in cat_cols if c not in empty_cols]

print('Fitur asli:',X_train.shape[1],'| Numerik:',len(num_cols),'| Kategorikal:',len(cat_cols))
print('Kategori:',cat_cols)
print('Kolom tidak digunakan:', sorted(EXCLUDE & set(train_df.columns)))

### Penjelasan Sel 7 — Strategi sampling dari Tugas 4
Pada Tugas 4, pilihan sampling ditentukan oleh rata-rata **Macro-F1 Cross-Validation**. Jika file `metadata.json` tugas keempat tersedia, notebook dapat membaca strategi terpilih itu. Jika tidak ada, secara transparan menggunakan **Tanpa sampling** sebagai baseline; kamu boleh mengganti variabel di bawah menjadi `'Undersampling'` atau `'Oversampling'` sesuai hasil Tugas 4.

Sampling diterapkan **hanya pada data training tiap fold**, bukan validation atau test.

In [ ]:
# 'auto' akan membaca metadata Tugas 4 jika tersedia; pilihan manual:
# 'Tanpa sampling', 'Undersampling', atau 'Oversampling'.
SAMPLING_STRATEGY = 'auto'
metadata_candidates = [
    WORK_DIR / 'Hasil_Validation_Sampling_UNSW_NB15' / 'metadata.json',
    Path('/content/drive/MyDrive/Hasil_Validation_Sampling_UNSW_NB15/metadata.json'),
]

if SAMPLING_STRATEGY == 'auto':
    old_meta = next((p for p in metadata_candidates if p.is_file()), None)
    if old_meta is not None:
        old_info = json.loads(old_meta.read_text(encoding='utf-8'))
        SAMPLING_STRATEGY = old_info.get('best_sampling_strategy_by_cv_macro_f1','Tanpa sampling')
        print('Strategi diambil dari:',old_meta)
    else:
        SAMPLING_STRATEGY = 'Tanpa sampling'
        print('Metadata Tugas 4 tidak tersedia. Menggunakan baseline Tanpa sampling.')

if SAMPLING_STRATEGY not in ['Tanpa sampling','Undersampling','Oversampling']:
    raise ValueError(f'Strategi tidak dikenal: {SAMPLING_STRATEGY}')
print('Sampling yang digunakan:', SAMPLING_STRATEGY)

## D. Pipeline dan baseline
### Penjelasan Sel 8 — Membuat pipeline tanpa data leakage
Urutannya: **imputasi median/modus → StandardScaler/OneHotEncoder → sampling opsional → SelectKBest → Logistic Regression**. Semuanya ditempatkan dalam satu pipeline agar setiap fold CV menghitung transformasi dari bagian training-nya sendiri.

- `SelectKBest(f_classif)` memberi skor pada setiap fitur untuk memilih fitur berkaitan dengan kelas Normal/Attack.
- `k='all'` berarti **tidak ada fitur yang dibuang** (baseline).
- `C` pada Logistic Regression mengontrol kekuatan regularisasi: **C kecil = regularisasi lebih kuat**.

In [ ]:
def buat_pipeline(k='all', C=1.0, class_weight=None):
    transform = []
    if num_cols:
        num_pipe = SkPipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])
        transform.append(('numerik',num_pipe,num_cols))
    if cat_cols:
        try:
            enc = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
        except TypeError:
            enc = OneHotEncoder(handle_unknown='ignore', sparse=True)
        cat_pipe = SkPipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', enc)
        ])
        transform.append(('kategori',cat_pipe,cat_cols))
    prep = ColumnTransformer(transform, remainder='drop', sparse_threshold=1.0)
    steps = [('prep', prep)]
    if SAMPLING_STRATEGY == 'Undersampling':
        steps.append(('sampling',RandomUnderSampler(random_state=RANDOM_STATE)))
    elif SAMPLING_STRATEGY == 'Oversampling':
        steps.append(('sampling',RandomOverSampler(random_state=RANDOM_STATE)))
    steps += [
        ('select',SelectKBest(score_func=f_classif,k=k)),
        ('model',LogisticRegression(solver='liblinear',max_iter=600,
                                    C=C,class_weight=class_weight,
                                    random_state=RANDOM_STATE))
    ]
    return ImbPipeline(steps)

def metrik(nama, y_true, y_pred):
    return {'Model':nama,
            'Accuracy':accuracy_score(y_true,y_pred),
            'Macro-F1':f1_score(y_true,y_pred,average='macro',zero_division=0),
            'Balanced Accuracy':balanced_accuracy_score(y_true,y_pred),
            'Precision Attack':precision_score(y_true,y_pred,pos_label=1,zero_division=0),
            'Recall Attack':recall_score(y_true,y_pred,pos_label=1,zero_division=0)}

print('Pipeline siap. Feature Selection dan sampling akan dipelajari dari train.')

### Penjelasan Sel 9 — Baseline tanpa Feature Selection
Kita melatih Logistic Regression menggunakan **semua fitur hasil encoding** (`k='all'`). Hasil prediksi pada validation menjadi **patokan**, supaya nanti terlihat apakah memilih fitur dan melakukan tuning benar-benar membantu. Model hanya dilatih pada training.

In [ ]:
baseline = buat_pipeline(k='all',C=1.0,class_weight=None)
baseline.fit(X_train,y_train)
pred_base = baseline.predict(X_val)
hasil_baseline = metrik('Baseline (semua fitur)', y_val, pred_base)

fitur_encoded = baseline.named_steps['prep'].get_feature_names_out()
n_fitur_encoded = len(fitur_encoded)
print('Jumlah fitur sesudah OneHotEncoding:',n_fitur_encoded)
display(pd.DataFrame([hasil_baseline]).round(4))

## E. Feature Selection
### Penjelasan Sel 10 — Mencoba SelectKBest
Kita memilih hingga **20 fitur dengan skor F tertinggi** (`k = min(20, jumlah fitur)`). Pemilihan dilakukan hanya pada data training. Lalu kita mengukur Macro-F1 validation dan membandingkannya dengan baseline.

**Catatan:** fitur setelah One-Hot Encoding dapat bernama `kategori__proto_tcp`, dan sebagainya; satu kolom kategori bisa menjadi beberapa fitur. Keterangan fitur terpilih bukan berarti fitur lain pasti tidak berguna.

In [ ]:
K_AWAL = min(20,n_fitur_encoded)
model_seleksi = buat_pipeline(k=K_AWAL,C=1.0,class_weight=None)
model_seleksi.fit(X_train,y_train)
pred_select = model_seleksi.predict(X_val)
hasil_seleksi = metrik(f'SelectKBest k={K_AWAL}',y_val,pred_select)

selector = model_seleksi.named_steps['select']
nama_fitur = model_seleksi.named_steps['prep'].get_feature_names_out()
skor = pd.DataFrame({'Fitur':nama_fitur,'F-score':selector.scores_,
                     'Terpilih':selector.get_support()})
fitur_terpilih = skor.loc[skor['Terpilih']].sort_values('F-score',ascending=False).reset_index(drop=True)

print('Jumlah fitur terpilih:',len(fitur_terpilih))
display(fitur_terpilih.head(20))
display(pd.DataFrame([hasil_baseline,hasil_seleksi]).round(4))

### Penjelasan Sel 11 — Visualisasi fitur terpilih
Grafik memperlihatkan fitur yang paling kuat menurut **F-score pada data training**. Semakin tinggi skor, semakin besar perbedaan nilai fitur antar kelas dalam pengujian statistik univariat ini.

Skor tinggi **tidak selalu** membuat model lebih akurat; karena itu tetap harus dibandingkan melalui Macro-F1.

In [ ]:
top = fitur_terpilih.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(10,6))
ax.barh(top['Fitur'],top['F-score'].fillna(0))
ax.set_title('15 Fitur Teratas — SelectKBest (Training)')
ax.set_xlabel('F-score (lebih tinggi ≠ jaminan lebih akurat)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'fitur_terpilih.png',dpi=150,bbox_inches='tight')
plt.show()

## F. Hyperparameter Tuning
### Penjelasan Sel 12 — GridSearchCV + Stratified K-Fold
`GridSearchCV` mencoba kombinasi pengaturan model secara sistematis. Pada setiap kombinasi, data **training** dibagi menjadi 3 fold berstrata. Model dipilih berdasarkan **rata-rata Macro-F1 Cross-Validation**, bukan dari test.

Yang dicoba:
- `select__k`: **20 fitur terbaik** atau **semua fitur**.
- `model__C`: **0.1, 1, 10** (kekuatan regularisasi).
- `model__class_weight`: **None** atau **balanced** (pemberian bobot kelas).

Total 2 × 3 × 2 = **12 kombinasi**, masing-masing 3 fold → **36 proses fit**. Waktu komputasi bergantung pada Colab. `n_jobs=1` dipilih agar penggunaan RAM lebih stabil.

In [ ]:
nilai_k = list(dict.fromkeys([K_AWAL, 'all']))
parameter_grid = {
    'select__k': nilai_k,
    'model__C': [0.1,1.0,10.0],
    'model__class_weight': [None,'balanced']
}
cv = StratifiedKFold(n_splits=FOLDS,shuffle=True,random_state=RANDOM_STATE)

pencarian = GridSearchCV(
    estimator=buat_pipeline(),
    param_grid=parameter_grid,
    scoring='f1_macro',
    cv=cv,
    refit=True,
    n_jobs=1,
    verbose=1,
    error_score='raise',
    return_train_score=False
)
pencarian.fit(X_train,y_train)

print('Parameter terbaik:',pencarian.best_params_)
print('Macro-F1 CV terbaik:',round(pencarian.best_score_,4))
model_terbaik = pencarian.best_estimator_

### Penjelasan Sel 13 — Membaca hasil GridSearchCV
Setiap baris tabel adalah satu kombinasi hyperparameter. Kolom `mean_test_score` berarti rata-rata **Macro-F1 dari 3 fold CV training** (kata `test` di sini berarti fold validasi internal CV, **bukan** test set akhir). Nilai lebih besar menunjukkan hasil yang lebih baik untuk ukuran ini.

In [ ]:
tabel_grid = pd.DataFrame(pencarian.cv_results_)
tabel_grid = tabel_grid[['rank_test_score','mean_test_score','std_test_score',
                          'param_select__k','param_model__C','param_model__class_weight']]
tabel_grid = tabel_grid.sort_values(['rank_test_score','mean_test_score'],
                                     ascending=[True,False]).reset_index(drop=True)
display(tabel_grid.round(4))

### Penjelasan Sel 14 — Perbandingan ketiga model pada validation
Model **baseline**, **SelectKBest awal**, dan **hasil tuning** diuji pada data validation yang sama. Dari tabel ini kita dapat menjelaskan apakah pemilihan fitur atau tuning berdampak pada performa.

Perbandingan ini bersifat konfirmasi; konfigurasi tuning sudah dipilih menggunakan CV training, bukan berdasarkan hasil test.

In [ ]:
pred_tuned_val = model_terbaik.predict(X_val)
hasil_tuned_val = metrik('GridSearchCV terbaik',y_val,pred_tuned_val)
perbandingan_validasi = pd.DataFrame([hasil_baseline,hasil_seleksi,hasil_tuned_val])
display(perbandingan_validasi.round(4))

ax = perbandingan_validasi.set_index('Model')[['Macro-F1','Balanced Accuracy','Recall Attack']].plot(
    kind='bar',figsize=(10,4),rot=15)
ax.set_title('Perbandingan pada Data Validation')
ax.set_ylim(0,1.05)
ax.set_ylabel('Skor (0–1)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'perbandingan_validasi.png',dpi=150,bbox_inches='tight')
plt.show()

## G. Evaluasi akhir pada test
### Penjelasan Sel 15 — Menguji model final
Sekarang baru kita melihat kinerja model terpilih pada **test set** yang tidak digunakan untuk memilih fitur maupun hyperparameter. **Macro-F1** merata-ratakan F1 untuk Normal dan Attack, **Balanced Accuracy** memberi bobot sama untuk kedua kelas, sedangkan **Recall Attack** menunjukkan bagian serangan yang berhasil ditemukan. Lihat juga confusion matrix untuk mengetahui salah klasifikasi.

In [ ]:
pred_test = model_terbaik.predict(X_test)
hasil_test = metrik('GridSearchCV terbaik — Test',y_test,pred_test)
display(pd.DataFrame([hasil_test]).round(4))
print('Classification Report (Test):')
print(classification_report(y_test,pred_test,
                            target_names=['Normal (0)','Attack (1)'],
                            digits=4,zero_division=0))

cm = confusion_matrix(y_test,pred_test,labels=[0,1])
TN,FP,FN,TP = cm.ravel()
print(f'TN={TN:,}, FP={FP:,}, FN={FN:,}, TP={TP:,}')
fig,ax=plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(cm,display_labels=['Normal','Attack']).plot(ax=ax,colorbar=False,values_format='d')
ax.set_title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'confusion_matrix_test.png',dpi=150,bbox_inches='tight')
plt.show()

### Penjelasan Sel 16 — Melihat fitur final setelah tuning
Karena `GridSearchCV` juga memilih `select__k`, jumlah fitur final bisa berbeda dari contoh awal k=20. Sel ini melaporkan **fitur yang benar-benar digunakan** oleh model terbaik. Jika `k='all'`, berarti hasil CV menganggap semua fitur lebih baik daripada hanya 20 fitur untuk grid yang diuji.

In [ ]:
selected_features_all = model_terbaik.named_steps['prep'].get_feature_names_out()
selected_mask = model_terbaik.named_steps['select'].get_support()
fitur_final = pd.DataFrame({
    'Fitur':selected_features_all,
    'F-score':model_terbaik.named_steps['select'].scores_,
    'Terpilih':selected_mask
})
fitur_final = fitur_final[fitur_final['Terpilih']].sort_values('F-score',ascending=False).reset_index(drop=True)
print('Jumlah fitur setelah encoding:',len(selected_features_all))
print('Jumlah fitur final:',len(fitur_final))
display(fitur_final.head(25))

## H. Pembahasan dan penyimpanan
### Penjelasan Sel 17 — Kesimpulan otomatis dari angka asli
Sel ini menulis ringkasan yang bersumber dari **hasil perhitungan Colab**, bukan angka buatan. Perhatikan apakah Macro-F1 tuning meningkat atau menurun dibanding baseline. Bila tuning tidak meningkat, jelaskan itu sebagai temuan; jangan mengklaim pasti lebih baik.

In [ ]:
delta = hasil_tuned_val['Macro-F1'] - hasil_baseline['Macro-F1']
arah = 'meningkat' if delta > 0 else ('menurun' if delta < 0 else 'tidak berubah')

laporan = f"""# Pembahasan Tugas 5 — UNSW-NB15

1. **Tujuan**: menentukan fitur yang relevan dan mencari hyperparameter terbaik untuk klasifikasi Normal (0) dan Attack (1).
2. **Dataset**: {TRAIN_FILE.name}; ukuran data train {len(X_train):,}, validation {len(X_val):,}, test {len(X_test):,}. Metode split: `{split_method}`.
3. **Preprocessing**: imputasi median pada numerik, modus dan One-Hot Encoding pada kategori, StandardScaler untuk numerik; di-fit hanya pada fold training. Kolom `id`, `attack_cat`, dan target `label` tidak dijadikan fitur.
4. **Sampling**: {SAMPLING_STRATEGY}; hanya pada training/fold CV.
5. **Feature Selection**: memakai `SelectKBest(f_classif)`; kandidat {nilai_k}; jumlah fitur final {len(fitur_final)} dari {len(selected_features_all)} hasil encoding.
6. **Tuning**: menggunakan GridSearchCV dengan {FOLDS} Stratified K-Fold, memilih berdasarkan Macro-F1. Parameter terbaik: `{pencarian.best_params_}`. Skor CV terbaik: {pencarian.best_score_:.4f}.
7. **Validasi**: Macro-F1 baseline {hasil_baseline['Macro-F1']:.4f} dan model tuning {hasil_tuned_val['Macro-F1']:.4f}; selisih {delta:+.4f} ({arah}).
8. **Evaluasi test**: Accuracy {hasil_test['Accuracy']:.4f}; Macro-F1 {hasil_test['Macro-F1']:.4f}; Balanced Accuracy {hasil_test['Balanced Accuracy']:.4f}; Precision Attack {hasil_test['Precision Attack']:.4f}; Recall Attack {hasil_test['Recall Attack']:.4f}.
9. **Keterbatasan**: hasil bergantung pada partisi, grid, dan karakteristik UNSW-NB15; bukan jaminan performa pada seluruh lalu lintas jaringan nyata. Test tidak digunakan untuk pemilihan fitur atau hyperparameter.
"""
display(Markdown(laporan))

### Penjelasan Sel 18 — Simpan artefak dan ZIP
Hasil yang disimpan: ringkasan kualitas data, split, perbandingan validation, hasil GridSearch, daftar fitur terpilih, metrik test, grafik, laporan, dan model `.joblib`. **Notebook `.ipynb` yang sudah dijalankan** tetap menjadi berkas utama untuk pengumpulan via link Google Colab.

In [ ]:
quality_report.to_csv(OUTPUT_DIR/'01_validasi_dataset.csv',index=False)
split_report.to_csv(OUTPUT_DIR/'02_split.csv',index=False)
perbandingan_validasi.to_csv(OUTPUT_DIR/'03_perbandingan_validation.csv',index=False)
tabel_grid.to_csv(OUTPUT_DIR/'04_hasil_gridsearch.csv',index=False)
fitur_final.to_csv(OUTPUT_DIR/'05_fitur_final.csv',index=False)
pd.DataFrame([hasil_test]).to_csv(OUTPUT_DIR/'06_metrik_test.csv',index=False)
(OUTPUT_DIR/'Penjelasan_Hasil.md').write_text(laporan,encoding='utf-8')
joblib.dump(model_terbaik,OUTPUT_DIR/'model_tugas5_logistic_regression.joblib',compress=3)

meta = {'dataset':'UNSW-NB15','source_train':TRAIN_FILE.name,
        'source_test':TEST_FILE.name if TEST_FILE else None,
        'split_method':split_method,'sampling':SAMPLING_STRATEGY,
        'target':'label','excluded_columns':sorted(EXCLUDE),
        'random_state':RANDOM_STATE,'cv_folds':FOLDS,
        'best_params':pencarian.best_params_,'cv_macro_f1':float(pencarian.best_score_),
        'validation_macro_f1':float(hasil_tuned_val['Macro-F1']),
        'test_macro_f1':float(hasil_test['Macro-F1']),
        'timestamp_utc':datetime.now(timezone.utc).isoformat()}
(OUTPUT_DIR/'metadata.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
zip_path = Path(shutil.make_archive(str(OUTPUT_DIR),'zip',root_dir=OUTPUT_DIR))
print('Selesai. Semua file tersimpan dalam:',OUTPUT_DIR)
print('ZIP:',zip_path)
for p in sorted(OUTPUT_DIR.iterdir()):print(' -',p.name)

## I. Cara mengumpulkan link Colab
1. Pastikan **sel terakhir selesai tanpa error** dan output hasil tuning/test muncul.
2. Pada Google Colab pilih **File → Save a copy in Drive** bila notebook belum tersimpan di Drive.
3. Klik **Share / Bagikan** → di **General access** pilih **Anyone with the link** → **Viewer** (sesuai kebijakan kelas), lalu **Copy link**.
4. Tempel **link Google Colab** pada kolom pengumpulan tugas kelima. Tidak perlu mengirim ZIP jika dosen hanya meminta link notebook.

**Materi yang perlu bisa dijelaskan:** mengapa `attack_cat` dikeluarkan; arti `SelectKBest`; apa yang diubah oleh `C` dan `class_weight`; mengapa `StratifiedKFold`; mengapa membandingkan dengan baseline; serta mengapa data test tidak dipakai tuning.